# Expanded RQ2 Embedding Job for Colab Pro

This notebook embeds the guarded expanded STEM sample for RQ2 using `sentence-transformers/all-MiniLM-L6-v2` on a Colab GPU. It preserves `expanded_row_id` order and writes resumable chunk outputs before assembling the final `.npy` file.

Upload only this local file to Google Drive:

- `data/q1_rq2_expanded_stem_sample.csv`

Recommended Drive path:

- `MyDrive/CreativeAI_RQ2/data/q1_rq2_expanded_stem_sample.csv`

Do not upload the full `arxiv_meta.csv`, old embeddings, old result CSVs, or legacy outputs for this job.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U sentence-transformers pyarrow

## Configuration

If you use a different Drive folder, change `PROJECT_DIR`. If Colab raises CUDA out-of-memory, reduce `BATCH_SIZE` to 256 or 128 and rerun the embedding cell; completed chunk files will be skipped.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/CreativeAI_RQ2')
SAMPLE_CSV = PROJECT_DIR / 'data' / 'q1_rq2_expanded_stem_sample.csv'
OUT_DIR = PROJECT_DIR / 'colab_outputs'
CHUNK_DIR = OUT_DIR / 'embedding_chunks'
FINAL_NPY = OUT_DIR / 'q1_rq2_expanded_embeddings.npy'
FINAL_MANIFEST = OUT_DIR / 'q1_rq2_expanded_embedding_manifest.json'

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
MODEL_REVISION = 'c9745ed1d9f207416be6d2e6f8de32d1f16199bf'
EXPECTED_ROWS = 425_728
EXPECTED_DIM = 384
BATCH_SIZE = 1024
CHUNK_ROWS = 20_000
REQUIRE_GPU = True

OUT_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

print('Sample CSV:', SAMPLE_CSV)
print('Output dir:', OUT_DIR)
print('Chunk dir:', CHUNK_DIR)

In [ ]:
import os
import json
import time
import hashlib
import numpy as np
import pandas as pd
import torch

if not SAMPLE_CSV.exists():
    raise FileNotFoundError(f'Missing sample CSV: {SAMPLE_CSV}')

gpu_available = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if gpu_available else 'CPU only'
print('Torch:', torch.__version__)
print('CUDA available:', gpu_available)
print('Device:', device_name)

if REQUIRE_GPU and not gpu_available:
    raise RuntimeError('GPU runtime is not active. In Colab, use Runtime > Change runtime type > GPU, then rerun.')

def file_sha256(path, block_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(block_size), b''):
            h.update(block)
    return h.hexdigest()

sample_size_bytes = SAMPLE_CSV.stat().st_size
sample_sha256 = file_sha256(SAMPLE_CSV)
print('Sample size bytes:', sample_size_bytes)
print('Sample SHA256:', sample_sha256)

## Load and Validate the Sample

The row order is sorted by `expanded_row_id`. This is the order expected by downstream local scripts.

In [ ]:
usecols = [
    'id',
    'expanded_row_id',
    'title',
    'abstract',
    'primary_category',
    'macro_field',
    'submission_month',
]

df = pd.read_csv(SAMPLE_CSV, usecols=usecols, dtype={'id': 'string'})
df = df.sort_values('expanded_row_id').reset_index(drop=True)

if len(df) != EXPECTED_ROWS:
    raise ValueError(f'Unexpected row count: {len(df)} != {EXPECTED_ROWS}')

expected_ids = np.arange(len(df), dtype=np.int64)
observed_ids = df['expanded_row_id'].to_numpy(dtype=np.int64)
if not np.array_equal(observed_ids, expected_ids):
    bad = int(np.where(observed_ids != expected_ids)[0][0])
    raise ValueError(f'expanded_row_id is not contiguous at position {bad}: {observed_ids[bad]}')

texts = (df['title'].fillna('').astype(str) + '. ' + df['abstract'].fillna('').astype(str)).tolist()
print('Rows:', len(df))
print('Primary categories:', df['primary_category'].nunique())
print('Macro fields:', df['macro_field'].nunique())
print('Month range:', df['submission_month'].min(), 'to', df['submission_month'].max())
print('First text length:', len(texts[0]))

## Embed in Resumable Chunks

Chunk files are stored in Drive under `colab_outputs/embedding_chunks`. If a chunk already exists with the correct shape, it is skipped.

In [ ]:
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer(MODEL_NAME, revision=MODEL_REVISION, device=device)
dim = int(model.get_sentence_embedding_dimension())

if dim != EXPECTED_DIM:
    raise ValueError(f'Unexpected embedding dimension: {dim} != {EXPECTED_DIM}')

def chunk_path(begin, end):
    return CHUNK_DIR / f'chunk_{begin:06d}_{end:06d}.npy'

def chunk_is_valid(path, rows, dim):
    if not path.exists():
        return False
    try:
        arr = np.load(path, mmap_mode='r')
        return arr.shape == (rows, dim) and arr.dtype == np.float32
    except Exception:
        return False

chunk_records = []
start_time = time.time()
total = len(texts)

for begin in range(0, total, CHUNK_ROWS):
    end = min(begin + CHUNK_ROWS, total)
    out_path = chunk_path(begin, end)
    rows = end - begin

    if chunk_is_valid(out_path, rows, dim):
        status = 'skipped_existing'
        print(f'Skip valid chunk {begin}:{end}')
    else:
        print(f'Embedding chunk {begin}:{end} on {device} with batch_size={BATCH_SIZE}')
        encoded = model.encode(
            texts[begin:end],
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        encoded = np.asarray(encoded, dtype=np.float32)
        if encoded.shape != (rows, dim):
            raise ValueError(f'Bad chunk shape for {begin}:{end}: {encoded.shape}')
        tmp_path = out_path.with_suffix('.tmp.npy')
        np.save(tmp_path, encoded)
        os.replace(tmp_path, out_path)
        status = 'embedded'

    chunk_records.append({
        'begin': int(begin),
        'end': int(end),
        'rows': int(rows),
        'path': str(out_path.relative_to(PROJECT_DIR)),
        'status': status,
    })
    progress_manifest = {
        'status': 'chunking_in_progress',
        'model': MODEL_NAME,
        'model_revision': MODEL_REVISION,
        'device': device,
        'device_name': device_name,
        'rows': int(total),
        'dimension': int(dim),
        'batch_size': int(BATCH_SIZE),
        'chunk_rows': int(CHUNK_ROWS),
        'chunks_completed_or_skipped': len(chunk_records),
        'chunks': chunk_records,
        'sample_csv': str(SAMPLE_CSV.relative_to(PROJECT_DIR)),
        'sample_size_bytes': int(sample_size_bytes),
        'sample_sha256': sample_sha256,
    }
    FINAL_MANIFEST.write_text(json.dumps(progress_manifest, indent=2))

elapsed = time.time() - start_time
print(f'Chunk embedding stage complete in {elapsed / 60:.2f} minutes')

## Assemble Final `.npy`

This creates `colab_outputs/q1_rq2_expanded_embeddings.npy`, the file to copy back into the local repository as `data/q1_rq2_expanded_embeddings.npy`.

In [ ]:
final = np.lib.format.open_memmap(FINAL_NPY, mode='w+', dtype='float32', shape=(total, dim))

for begin in range(0, total, CHUNK_ROWS):
    end = min(begin + CHUNK_ROWS, total)
    path = chunk_path(begin, end)
    if not chunk_is_valid(path, end - begin, dim):
        raise FileNotFoundError(f'Missing or invalid chunk: {path}')
    arr = np.load(path, mmap_mode='r')
    final[begin:end, :] = arr
    final.flush()
    print(f'Assembled {end}/{total}')

del final

check = np.load(FINAL_NPY, mmap_mode='r')
if check.shape != (EXPECTED_ROWS, EXPECTED_DIM):
    raise ValueError(f'Final embedding shape mismatch: {check.shape}')

final_manifest = {
    'script': 'colab_embed_expanded_rq2_sample.ipynb',
    'status': 'complete',
    'model': MODEL_NAME,
    'model_revision': MODEL_REVISION,
    'device': device,
    'device_name': device_name,
    'sample': str(SAMPLE_CSV.relative_to(PROJECT_DIR)),
    'sample_size_bytes': int(sample_size_bytes),
    'sample_sha256': sample_sha256,
    'output': str(FINAL_NPY.relative_to(PROJECT_DIR)),
    'rows': int(EXPECTED_ROWS),
    'dimension': int(EXPECTED_DIM),
    'dtype': 'float32',
    'batch_size': int(BATCH_SIZE),
    'chunk_rows': int(CHUNK_ROWS),
    'chunks': chunk_records,
}
FINAL_MANIFEST.write_text(json.dumps(final_manifest, indent=2))

print('Final array:', FINAL_NPY)
print('Final shape:', check.shape)
print('Final manifest:', FINAL_MANIFEST)

## Files to Bring Back to the Local Repository

Copy these completed Colab outputs back to the local project:

- `MyDrive/CreativeAI_RQ2/colab_outputs/q1_rq2_expanded_embeddings.npy` -> local `data/q1_rq2_expanded_embeddings.npy`
- `MyDrive/CreativeAI_RQ2/colab_outputs/q1_rq2_expanded_embedding_manifest.json` -> local `results/q1_rq2_expanded_embedding_manifest.json`

The folder `MyDrive/CreativeAI_RQ2/colab_outputs/embedding_chunks/` can stay in Drive as checkpoint backup. It is not needed locally after the final `.npy` is complete.